# Data Wrangling: Cleaning, Encoding and Feature Engineering

Part of my IBM Data Analyst Capstone project. In this notebook I prepare the Stack Overflow developer survey for analysis: explore the data, standardise inconsistent text, encode a categorical column, handle missing values, reduce skew in compensation, and create a new experience-level feature.

This version of the survey file has already had its duplicates removed (65,437 rows). Duplicate handling and Min-Max scaling are covered in the earlier notebooks.

In [1]:
import pandas as pd
import numpy as np

## 1. Load the data

In [2]:
# Load the Stack Overflow survey data
dataset_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/n01PQ9pSmiRX6520flujwQ/survey-data.csv"
df = pd.read_csv(dataset_url)

# Display the first few rows
print(df.head())

   ResponseId                      MainBranch                 Age  \
0           1  I am a developer by profession  Under 18 years old   
1           2  I am a developer by profession     35-44 years old   
2           3  I am a developer by profession     45-54 years old   
3           4           I am learning to code     18-24 years old   
4           5  I am a developer by profession     18-24 years old   

            Employment RemoteWork   Check  \
0  Employed, full-time     Remote  Apples   
1  Employed, full-time     Remote  Apples   
2  Employed, full-time     Remote  Apples   
3   Student, full-time        NaN  Apples   
4   Student, full-time        NaN  Apples   

                                    CodingActivities  \
0                                              Hobby   
1  Hobby;Contribute to open-source projects;Other...   
2  Hobby;Contribute to open-source projects;Other...   
3                                                NaN   
4                                 

## 2. Explore the data
Column types, non-null counts and missing values.

In [3]:
# 2.1 Data types, counts, and missing values
df.info()

# Or as a combined summary table
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non_null': df.count(),
    'missing': df.isnull().sum()
})
print(summary)

<class 'pandas.DataFrame'>
RangeIndex: 65437 entries, 0 to 65436
Columns: 114 entries, ResponseId to JobSat
dtypes: float64(13), int64(1), str(100)
memory usage: 195.7 MB
                       dtype  non_null  missing
ResponseId             int64     65437        0
MainBranch               str     65437        0
Age                      str     65437        0
Employment               str     65437        0
RemoteWork               str     54806    10631
...                      ...       ...      ...
JobSatPoints_11      float64     29445    35992
SurveyLength             str     56182     9255
SurveyEase               str     56238     9199
ConvertedCompYearly  float64     23435    42002
JobSat               float64     29126    36311

[114 rows x 3 columns]

Summary statistics for the numerical columns.

In [4]:
# 2.2 Basic statistics for numerical columns
print(df.describe())

         ResponseId      CompTotal       WorkExp  JobSatPoints_1  \
count  65437.000000   3.374000e+04  29658.000000    29324.000000   
mean   32719.000000  2.963841e+145     11.466957       18.581094   
std    18890.179119  5.444117e+147      9.168709       25.966221   
min        1.000000   0.000000e+00      0.000000        0.000000   
25%    16360.000000   6.000000e+04      4.000000        0.000000   
50%    32719.000000   1.100000e+05      9.000000       10.000000   
75%    49078.000000   2.500000e+05     16.000000       22.000000   
max    65437.000000  1.000000e+150     50.000000      100.000000   

       JobSatPoints_4  JobSatPoints_5  JobSatPoints_6  JobSatPoints_7  \
count    29393.000000    29411.000000    29450.000000     29448.00000   
mean         7.522140       10.060857       24.343232        22.96522   
std         18.422661       21.833836       27.089360        27.01774   
min          0.000000        0.000000        0.000000         0.00000   
25%          0.000000 

`CompTotal` has a mean of about 2.96 × 10¹⁴⁵, which is impossible, so it contains entry errors. This is why I use `ConvertedCompYearly` (annual compensation converted to USD) for pay analysis.

## 3. Standardise inconsistent entries
Check how many distinct countries there are, then shorten the long country and education labels.

In [5]:
# 3.1 Look for inconsistent entries in Country
print(df['Country'].nunique())
print(df['Country'].value_counts())

# Show the unique values sorted, to spot odd spellings or long names
print(sorted(df['Country'].dropna().unique()))

185
Country
United States of America                                11095
Germany                                                  4947
India                                                    4231
United Kingdom of Great Britain and Northern Ireland     3224
Ukraine                                                  2672
                                                        ...  
Haiti                                                       1
Nauru                                                       1
Chad                                                        1
Djibouti                                                    1
Solomon Islands                                             1
Name: count, Length: 185, dtype: int64
['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas', 'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium', 'Belize', 'Benin', 'Bhutan', 'Bolivia', 'Bosnia and H

In [6]:
# 3.2 Standardize entries by mapping inconsistent values to a consistent format
country_map = {
    'United States of America': 'USA',
    'United Kingdom of Great Britain and Northern Ireland': 'UK',
    'Russian Federation': 'Russia',
    'Iran, Islamic Republic of...': 'Iran',
    'Hong Kong (S.A.R.)': 'Hong Kong',
    'Republic of Korea': 'South Korea',
}
df['Country'] = df['Country'].replace(country_map)

# Also clean stray whitespace and casing
df['Country'] = df['Country'].str.strip()

# EdLevel: shorten the long labels
edlevel_map = {
    'Bachelor’s degree (B.A., B.S., B.Eng., etc.)': 'Bachelor’s',
    'Master’s degree (M.A., M.S., M.Eng., MBA, etc.)': 'Master’s',
    'Primary/elementary school': 'Primary',
}
df['EdLevel'] = df['EdLevel'].replace(edlevel_map)

print(df['Country'].value_counts().head(10))

Country
USA            11095
Germany         4947
India           4231
UK              3224
Ukraine         2672
France          2110
Canada          2104
Poland          1534
Netherlands     1449
Brazil          1375
Name: count, dtype: int64

## 4. Encode a categorical column
One-hot encode `Employment`. Respondents can pick several employment types, so I split the answers on ";" to get one yes/no column per type.

In [ ]:
# Employment holds several answers separated by ";", so I split on ";"
# to get one column per employment type.
employment_encoded = df['Employment'].str.get_dummies(sep=';').add_prefix('Employment_')
df = pd.concat([df, employment_encoded], axis=1)

print(employment_encoded.columns.tolist())
print(employment_encoded.head())

## 5. Handle missing values
The columns with the most missing values.

In [7]:
# 5.1 Columns with the most missing values
missing = df.isnull().sum().sort_values(ascending=False)
print(missing.head(10))

AINextMuch less integrated       64289
AINextLess integrated            63082
AINextNo change                  52939
AINextMuch more integrated       51999
EmbeddedAdmired                  48704
EmbeddedWantToWorkWith           47837
EmbeddedHaveWorkedWith           43223
ConvertedCompYearly              42002
AIToolNot interested in Using    41023
AINextMore integrated            41009
dtype: int64

About 64% of respondents did not report compensation. Filling those gaps with the median would give most rows an invented salary and flatten the real spread, so I leave them missing and analyse only respondents who reported pay (`comp_df`).

In [ ]:
missing_comp = df['ConvertedCompYearly'].isnull().sum()
print("Missing in ConvertedCompYearly:", missing_comp, f"({missing_comp / len(df):.1%})")

# Keep only respondents who reported compensation
comp_df = df.dropna(subset=['ConvertedCompYearly']).copy()
print("Rows with reported compensation:", len(comp_df))

`RemoteWork` is categorical, so I fill its gaps with the most frequent value.

In [8]:
# 5.3 Impute categorical column with the most frequent value (mode)
mode_remote = df['RemoteWork'].mode()[0]
df['RemoteWork'] = df['RemoteWork'].fillna(mode_remote)

print("Missing after imputation:", df['RemoteWork'].isnull().sum())

Missing after imputation: 0

## 6. Reduce skew in compensation
Compensation is heavily right-skewed. A log transform (`log1p`) pulls in the extreme values. Min-Max scaling and Z-scores are covered in the normalisation notebook.

In [ ]:
# log1p = log(1 + x), which safely handles zeros
comp_df['ConvertedCompYearly_Log'] = np.log1p(comp_df['ConvertedCompYearly'])

print("Skew before:", comp_df['ConvertedCompYearly'].skew())
print("Skew after: ", comp_df['ConvertedCompYearly_Log'].skew())

## 7. Feature engineering: experience level
Convert `YearsCodePro` to numbers, then group respondents into four experience levels.

In [ ]:
# YearsCodePro contains text values that must be converted to numbers first
df['YearsCodePro'] = df['YearsCodePro'].replace({
    'Less than 1 year': 0.5,
    'More than 50 years': 51
})
df['YearsCodePro'] = pd.to_numeric(df['YearsCodePro'], errors='coerce')

# Bin into experience levels
df['ExperienceLevel'] = pd.cut(
    df['YearsCodePro'],
    bins=[-1, 2, 5, 10, float('inf')],
    labels=['Junior (0-2)', 'Mid (3-5)', 'Senior (6-10)', 'Expert (11+)']
)

print(df['ExperienceLevel'].value_counts())